In [1]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio

# ============================
# Γ: espiral logarítmica  t = r e^{i k log r}
# ============================
def phi(r, k=3.2, theta0=0.0):
    r = np.asarray(r)
    theta = theta0 + k * np.log(r)
    return r * np.exp(1j * theta)

# ============================
# a_phi(r): ejemplo slowly oscillating (para visual) en r->∞
# ============================
def a_phi(r):
    r = np.asarray(r)
    # enfocado en r->∞
    return np.sin(np.log(np.log(np.e + r)))

def fig_to_rgb(fig):
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    return rgba[..., :3].copy()

def window_stats(r0, n=900):
    xs = np.linspace(r0, 2*r0, n)
    vals = a_phi(xs)
    i_min = int(np.argmin(vals))
    i_max = int(np.argmax(vals))
    vmin = float(vals[i_min])
    vmax = float(vals[i_max])
    return xs, vals, xs[i_min], vmin, xs[i_max], vmax, (vmax - vmin)

def add_clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

def make_gif_to_infty(
    out_path="spiral_with_osc_to_infty_row.gif",
    seconds_per_frame=0.18,   # más grande = más lento
    loop=0,
    k=3.2,
    dpi=200,                 # más grande = más resolución
    figsize=(13.8, 5.4),
    n_frames=95,
):
    fps = max(1, int(round(1.0 / seconds_per_frame)))

    # Rango global para la espiral (fijo) ahora con radios grandes
    r_min_all, r_max_all = 1e-3, 80.0
    r_all = np.logspace(np.log10(r_min_all), np.log10(r_max_all), 7000)
    z_all = phi(r_all, k=k)

    # Frames: r crece hacia "∞" (en la práctica: hacia r_max_all/2)
    r_start = 0.08
    r_end = r_max_all / 2.2          # para que [r,2r] quede dentro del rango
    r_path = np.geomspace(r_start, r_end, n_frames)

    fig, (ax_sp, ax_fn) = plt.subplots(
        1, 2, figsize=figsize, dpi=dpi, gridspec_kw={"wspace": 0.20}
    )

    frames = []

    for r0 in r_path:
        rr = np.linspace(r0, 2*r0, 900)
        rr = rr[(rr >= r_min_all) & (rr <= r_max_all)]
        if len(rr) < 5:
            continue
        z_seg = phi(rr, k=k)

        xs, vals, x_min, vmin, x_max, vmax, Delta = window_stats(r0)

        # -------------------------
        # Panel izquierdo: espiral
        # -------------------------
        ax_sp.clear()
        ax_sp.grid(True, linewidth=0.6, alpha=0.25)

        ax_sp.plot(z_all.real, z_all.imag, lw=2.5, alpha=0.22)
        ax_sp.plot(z_seg.real, z_seg.imag, lw=6.0, alpha=0.95)

        ax_sp.scatter(
            [z_seg[0].real, z_seg[-1].real],
            [z_seg[0].imag, z_seg[-1].imag],
            s=90,
            zorder=5,
        )

        ax_sp.set_aspect("equal", adjustable="datalim")
        add_clean_axes(ax_sp)
        ax_sp.set_title(
            rf"Espiral $\Gamma$  ($r\to\infty$ en rango finito,  $r={r0:.3g}$)",
            fontsize=12
        )

        # -------------------------
        # Panel derecho: a_phi
        # -------------------------
        ax_fn.clear()
        ax_fn.grid(True, linewidth=0.6, alpha=0.25)

        ax_fn.plot(xs, vals, lw=2.3)
        ax_fn.scatter([x_min, x_max], [vmin, vmax], s=70, zorder=6)

        x_mid = np.sqrt(r0 * 2*r0)
        ax_fn.plot([x_mid, x_mid], [vmin, vmax], lw=2.8)

        ax_fn.set_xscale("log")
        ax_fn.set_xlabel(r"$x\in[r,2r]$")
        ax_fn.set_ylabel(r"$a_\phi(x)$")

        ax_fn.text(
            0.02, 0.95,
            rf"$\Delta(r)=\max a_\phi-\min a_\phi={Delta:.3f}$",
            transform=ax_fn.transAxes,
            fontsize=11,
            va="top"
        )
        ax_fn.text(
            0.02, 0.82,
            r"$\Delta(r)=\sup_{x,y\in[r,2r]}|a_\phi(x)-a_\phi(y)|$",
            transform=ax_fn.transAxes,
            fontsize=11,
            va="top"
        )

        for sp in ax_fn.spines.values():
            sp.set_visible(False)

        frames.append(fig_to_rgb(fig))

    plt.close(fig)

    imageio.mimsave(out_path, frames, fps=fps, loop=loop)
    print(f"[OK] {out_path} | fps={fps} (~{seconds_per_frame:.2f}s/frame) | dpi={dpi} | loop={'inf' if loop==0 else loop}")

if __name__ == "__main__":
    make_gif_to_infty(
        out_path="spiral_with_osc_to_infty_row.gif",
        seconds_per_frame=0.18,
        dpi=300,
        n_frames=105,
    )

[OK] spiral_with_osc_to_infty_row.gif | fps=6 (~0.18s/frame) | dpi=300 | loop=inf
